In [1]:
import subprocess
subprocess.run(['pip', 'install', 'nest_asyncio'], capture_output=True)
print('done')

done


In [2]:
!pip install python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: C:\Users\PMLS\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [ ]:

import os, sys, csv, asyncio, logging, time, json
from datetime import datetime, date
from pathlib import Path
from functools import wraps
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

env_path = Path('.env')
if not env_path.exists():
    raise FileNotFoundError('❌ .env file not found. Please create one.')
load_dotenv(env_path)


PORTAL_NAME     = os.getenv('PORTAL_NAME', 'Academic Portal')
PASS_MARK       = float(os.getenv('PASS_MARK', 50))
FINE_PER_DAY    = float(os.getenv('LATE_FINE_PER_DAY', 50.0))
MAX_FINE        = float(os.getenv('MAX_FINE_CAP', 150.0))
REPORT_DIR      = os.getenv('REPORT_OUTPUT_DIR', 'reports')
LOG_FILE        = os.getenv('LOG_FILE', 'portal_errors.log')
ADMIN_CODE      = os.getenv('ADMIN_CODE', 'ADMIN2024')

Path(REPORT_DIR).mkdir(exist_ok=True)
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.ERROR,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('AcademicPortal')

print(f'✅ Environment loaded — {PORTAL_NAME}')
print(f'   Pass mark: {PASS_MARK} | Fine/day: rupees{FINE_PER_DAY} | Max fine: rupees{MAX_FINE}')
print(f'   Reports → ./{REPORT_DIR}/ | Errors → {LOG_FILE}')

✅ Environment loaded — University Academic Portal
   Pass mark: 50.0 | Fine/day: rupees2.5 | Max fine: rupees100.0
   Reports → ./reports/ | Errors → portal_errors.log


In [ ]:

class PortalBaseException(Exception):
    """Root exception for all Academic Portal errors."""
    def __init__(self, message: str, student_id: str = None):
        self.student_id = student_id
        self.timestamp  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        super().__init__(message)

    def log(self):
        logger.error(f'[{type(self).__name__}] student={self.student_id} | {self}')


class ValidationError(PortalBaseException):
    """Raised when a student record fails field validation."""

class MissingFieldError(ValidationError):
    """Raised when a required CSV field is absent or empty."""
    def __init__(self, field_name: str, student_id: str = None):
        self.field_name = field_name
        super().__init__(f"Missing required field: '{field_name}'", student_id)

class InvalidGradeError(ValidationError):
    """Raised when a grade value is outside 0–100."""
    def __init__(self, grade_val, student_id: str = None):
        super().__init__(f"Grade '{grade_val}' is not in range 0–100", student_id)

class CSVLoadError(PortalBaseException):
    """Raised when a CSV file cannot be read or parsed."""

class FineCalculationError(PortalBaseException):
    """Raised when date parsing fails during fine calculation."""

class AdminAuthError(PortalBaseException):
    """Raised when admin authentication fails."""

print('✅ Custom exception hierarchy defined')
print('   PortalBaseException')
print('   ├── ValidationError')
print('   │   ├── MissingFieldError')
print('   │   └── InvalidGradeError')
print('   ├── CSVLoadError')
print('   ├── FineCalculationError')
print('   └── AdminAuthError')

✅ Custom exception hierarchy defined
   PortalBaseException
   ├── ValidationError
   │   ├── MissingFieldError
   │   └── InvalidGradeError
   ├── CSVLoadError
   ├── FineCalculationError
   └── AdminAuthError


In [ ]:


def log_errors(func):
    """Decorator: catches PortalBaseExceptions, logs them, and re-raises."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PortalBaseException as e:
            e.log()
            raise
        except Exception as e:
            logger.error(f'[Unhandled in {func.__name__}] {e}')
            raise
    return wrapper


def async_log_errors(func):
    """Async version of log_errors decorator."""
    @wraps(func)
    async def wrapper(*args, **kwargs):
        try:
            return await func(*args, **kwargs)
        except PortalBaseException as e:
            e.log()
            raise
        except Exception as e:
            logger.error(f'[Unhandled async in {func.__name__}] {e}')
            raise
    return wrapper


def timer(func):
    """Decorator: prints execution time of any function."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'   ⏱  {func.__name__} completed in {elapsed:.4f}s')
        return result
    return wrapper


def validate_admin(func):
    """Decorator: prompts for admin code before executing sensitive operations."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        code = input('🔐 Enter admin code: ').strip()
        if code != ADMIN_CODE:
            err = AdminAuthError('Invalid admin code provided')
            err.log()
            print('❌ Access denied. Action logged.')
            return None
        print('✅ Admin authenticated.')
        return func(*args, **kwargs)
    return wrapper


print('✅ Decorators defined: @log_errors, @async_log_errors, @timer, @validate_admin')

✅ Decorators defined: @log_errors, @async_log_errors, @timer, @validate_admin


In [ ]:


@dataclass
class Student:
    student_id:       str
    name:             str
    email:            str
    course:           str
    grade:            Optional[float]
    submission_date:  str
    due_date:         str
    tuition_balance:  float
    late_fine:        float = field(default=0.0, init=False)
    days_late:        int   = field(default=0,   init=False)
    status:           str   = field(default='',  init=False)
    validation_errors: list = field(default_factory=list, init=False)

    def __post_init__(self):
        self.status = self._compute_status()

    def _compute_status(self) -> str:
        if self.grade is None:
            return 'PENDING'
        return 'PASS' if self.grade >= PASS_MARK else 'FAIL'

    def is_valid(self) -> bool:
        return len(self.validation_errors) == 0

    def summary(self) -> str:
        grade_str = f'{self.grade:.1f}' if self.grade is not None else 'N/A'
        fine_str  = f'rupees{self.late_fine:.2f}' if self.late_fine > 0 else 'None'
        return (f'{self.student_id:<6} | {self.name:<20} | {self.course:<12} | '
                f'Grade: {grade_str:>5} | {self.status:<7} | '
                f'Late: {self.days_late}d | Fine: {fine_str}')


class StudentValidator:
    """Validates Student records and returns a list of errors found."""

    REQUIRED_FIELDS = ['student_id', 'name', 'email', 'course',
                        'submission_date', 'due_date']

    @staticmethod
    @log_errors
    def validate(student: Student) -> list:
        errors = []

        # Required fields
        for f in StudentValidator.REQUIRED_FIELDS:
            val = getattr(student, f, None)
            if not val or (isinstance(val, str) and not val.strip()):
                err = MissingFieldError(f, student.student_id)
                errors.append(str(err))

        # Grade range
        if student.grade is not None:
            if not (0 <= student.grade <= 100):
                err = InvalidGradeError(student.grade, student.student_id)
                errors.append(str(err))

        # Email format
        if student.email and '@' not in student.email:
            errors.append(f"Invalid email format: '{student.email}'")

        student.validation_errors = errors
        return errors


print('✅ Student dataclass and StudentValidator defined')
print('   Validates: required fields, grade range (0-100), email format')

✅ Student dataclass and StudentValidator defined
   Validates: required fields, grade range (0-100), email format


In [ ]:


class FineCalculator:
    """Calculates late submission fines based on .env config."""

    def __init__(self, fine_per_day: float = FINE_PER_DAY, max_fine: float = MAX_FINE):
        self.fine_per_day = fine_per_day
        self.max_fine     = max_fine

    @log_errors
    def calculate(self, student: Student) -> float:
        try:
            sub_date = datetime.strptime(student.submission_date, '%Y-%m-%d').date()
            due_date = datetime.strptime(student.due_date,        '%Y-%m-%d').date()
        except ValueError as e:
            raise FineCalculationError(
                f'Date parse error: {e}', student.student_id
            )

        days_late = (sub_date - due_date).days
        if days_late <= 0:
            student.days_late = 0
            student.late_fine = 0.0
            return 0.0

        fine = min(days_late * self.fine_per_day, self.max_fine)
        student.days_late = days_late
        student.late_fine = fine
        return fine

    def process_all(self, students: list) -> dict:
        """Calculate fines for all students and return summary stats."""
        total_fines  = 0.0
        late_count   = 0
        for s in students:
            fine = self.calculate(s)
            total_fines += fine
            if fine > 0:
                late_count += 1
        return {'total_fines': total_fines, 'late_submissions': late_count}


print('✅ FineCalculator defined')
print(f'   Rate: rupees{FINE_PER_DAY}/day | Cap: rupees{MAX_FINE}')

✅ FineCalculator defined
   Rate: rupees2.5/day | Cap: rupees100.0


In [ ]:


class CSVLoader:
    """Asynchronously loads and parses student CSV files."""

    def __init__(self, filepath: str):
        self.filepath = filepath

    @async_log_errors
    async def load(self) -> list:
        """Async CSV loading — simulates non-blocking I/O."""
        if not Path(self.filepath).exists():
            raise CSVLoadError(f"File not found: '{self.filepath}'")

        students = []
        skipped  = []

        await asyncio.sleep(0)  # yield to event loop (simulates async I/O)

        try:
            with open(self.filepath, newline='', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for i, row in enumerate(reader, start=2):
                    try:
                        grade_raw = row.get('grade', '').strip()
                        grade = float(grade_raw) if grade_raw else None

                        student = Student(
                            student_id      = row.get('student_id', '').strip(),
                            name            = row.get('name', '').strip(),
                            email           = row.get('email', '').strip(),
                            course          = row.get('course', '').strip(),
                            grade           = grade,
                            submission_date = row.get('submission_date', '').strip(),
                            due_date        = row.get('due_date', '').strip(),
                            tuition_balance = float(row.get('tuition_balance', 0) or 0),
                        )
                        students.append(student)

                    except (ValueError, KeyError) as e:
                        skipped.append((i, str(e)))
                        err = CSVLoadError(f'Row {i} skipped: {e}')
                        err.log()

        except Exception as e:
            raise CSVLoadError(f'Failed to read CSV: {e}')

        print(f'   📂 Loaded {len(students)} records from {self.filepath}')
        if skipped:
            print(f'   ⚠️  Skipped {len(skipped)} malformed rows → logged to {LOG_FILE}')

        return students


print('✅ CSVLoader (async) defined')

✅ CSVLoader (async) defined


In [ ]:


class ReportGenerator:
    """Generates timestamped student reports asynchronously."""

    def __init__(self, output_dir: str = REPORT_DIR):
        self.output_dir = output_dir
        Path(output_dir).mkdir(exist_ok=True)

    def _timestamp(self) -> str:
        return datetime.now().strftime('%Y%m%d_%H%M%S')

    def _build_report(self, students: list, title: str) -> str:
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        total   = len(students)
        passed  = sum(1 for s in students if s.status == 'PASS')
        failed  = sum(1 for s in students if s.status == 'FAIL')
        pending = sum(1 for s in students if s.status == 'PENDING')
        invalid = sum(1 for s in students if not s.is_valid())
        total_fines = sum(s.late_fine for s in students)
        total_balance = sum(s.tuition_balance for s in students)

        lines = [
            '=' * 90,
            f'  {PORTAL_NAME}',
            f'  Report: {title}',
            f'  Generated: {now}',
            '=' * 90,
            '',
            '── SUMMARY ──────────────────────────────────────────────────────────────────────',
            f'  Total Students   : {total}',
            f'  Passed           : {passed}  ({passed/total*100:.1f}%)' if total else '  Passed: 0',
            f'  Failed           : {failed}  ({failed/total*100:.1f}%)' if total else '  Failed: 0',
            f'  Pending (no grade): {pending}',
            f'  Invalid Records  : {invalid}',
            f'  Total Late Fines : £{total_fines:.2f}',
            f'  Total Tuition Owed: £{total_balance:.2f}',
            '',
        ]
        failed_list = [s for s in students if s.status == 'FAIL']
        if failed_list:
            lines.append('── FAILED STUDENTS ─────────────────────────────────────────────────────────────')
            lines.append(f'  {"ID":<6} {"Name":<22} {"Course":<14} {"Grade":>6} {"Fine":>8} {"Balance":>10}')
            lines.append('  ' + '-' * 74)
            for s in failed_list:
                grade_str = f'{s.grade:.1f}' if s.grade is not None else 'N/A'
                lines.append(f'  {s.student_id:<6} {s.name:<22} {s.course:<14} '
                              f'{grade_str:>6} {"£"+str(round(s.late_fine,2)):>8} '
                              f'{"£"+str(round(s.tuition_balance,2)):>10}')
            lines.append('')
        invalid_list = [s for s in students if not s.is_valid()]
        if invalid_list:
            lines.append('── VALIDATION ERRORS ───────────────────────────────────────────────────────────')
            for s in invalid_list:
                lines.append(f'  [{s.student_id}] {s.name or "(no name)"}')
                for err in s.validation_errors:
                    lines.append(f'      ⚠ {err}')
            lines.append('')

        lines.append('── ALL RECORDS ─────────────────────────────────────────────────────────────────')
        lines.append(f'  {"ID":<6} {"Name":<22} {"Course":<14} {"Grade":>6} {"Status":<8} {"Late":>5} {"Fine":>8}')
        lines.append('  ' + '-' * 82)
        for s in students:
            grade_str = f'{s.grade:.1f}' if s.grade is not None else 'N/A'
            lines.append(f'  {s.student_id:<6} {s.name:<22} {s.course:<14} '
                          f'{grade_str:>6} {s.status:<8} {str(s.days_late)+"d":>5} '
                          f'{"£"+str(round(s.late_fine,2)):>8}')

        lines.append('')
        lines.append('=' * 90)
        lines.append(f'  End of Report — {PORTAL_NAME}')
        lines.append('=' * 90)
        return '\n'.join(lines)

    async def generate(self, students: list, report_type: str = 'full') -> str:
        """Async report generation with file write."""
        await asyncio.sleep(0)  # yield to event loop
        title    = f'{report_type.upper()} STUDENT REPORT'
        content  = self._build_report(students, title)
        filename = f'report_{report_type}_{self._timestamp()}.txt'
        filepath = Path(self.output_dir) / filename
        filepath.write_text(content, encoding='utf-8')
        return str(filepath)

    async def generate_failed_report(self, students: list) -> str:
        failed = [s for s in students if s.status == 'FAIL']
        return await self.generate(failed, 'failed_only')

    async def generate_fines_report(self, students: list) -> str:
        fined = [s for s in students if s.late_fine > 0]
        return await self.generate(fined, 'fines_only')


print('✅ ReportGenerator (async) defined')
print(f'   Reports saved to: ./{REPORT_DIR}/')

✅ ReportGenerator (async) defined
   Reports saved to: ./reports/


In [ ]:

class AcademicPortal:
    """Main controller — orchestrates loading, validation, fines, and reporting."""

    def __init__(self):
        self.students:    list           = []
        self.validator:   StudentValidator  = StudentValidator()
        self.calculator:  FineCalculator    = FineCalculator()
        self.reporter:    ReportGenerator   = ReportGenerator()
        self._loaded:     bool           = False

    async def load_csv(self, filepath: str):
        """Step 1: Load CSV asynchronously."""
        loader = CSVLoader(filepath)
        self.students = await loader.load()
        self._loaded  = True

    async def validate_all(self):
        """Step 2: Validate every student record concurrently (via gather)."""
        await asyncio.gather(*[
            asyncio.to_thread(StudentValidator.validate, s)
            for s in self.students
        ])

    async def calculate_fines(self):
        """Step 3: Compute late fines for all students concurrently."""
        await asyncio.gather(*[
            asyncio.to_thread(self.calculator.calculate, s)
            for s in self.students
        ])

    async def generate_reports(self) -> list:
        """Step 4: Generate 3 reports concurrently."""
        paths = await asyncio.gather(
            self.reporter.generate(self.students, 'full'),
            self.reporter.generate_failed_report(self.students),
            self.reporter.generate_fines_report(self.students),
        )
        return list(paths)

    @timer
    def run_pipeline(self, filepath: str):
        """Full pipeline — load → validate → fines → reports."""
        async def _pipeline():
            print('\n' + '─'*60)
            print(f'🎓 {PORTAL_NAME} — Processing Pipeline')
            print('─'*60)

            print('\n[1/4] 📂 Loading CSV...')
            await self.load_csv(filepath)

            print('[2/4] 🔍 Validating records (async concurrent)...')
            await self.validate_all()
            invalid = [s for s in self.students if not s.is_valid()]
            print(f'   Found {len(invalid)} invalid record(s) → errors logged to {LOG_FILE}')

            print('[3/4] 💰 Calculating late fines (async concurrent)...')
            await self.calculate_fines()
            stats = self.calculator.process_all([])
            total_fines = sum(s.late_fine for s in self.students)
            late_count  = sum(1 for s in self.students if s.late_fine > 0)
            print(f'   {late_count} late submission(s), total fines: rupees{total_fines:.2f}')

            print('[4/4] 📄 Generating reports (async concurrent)...')
            paths = await self.generate_reports()
            for p in paths:
                print(f'   ✅ Saved: {p}')

            return paths

        return asyncio.get_event_loop().run_until_complete(_pipeline())

    def get_failed_students(self) -> list:
        return [s for s in self.students if s.status == 'FAIL']

    def get_invalid_records(self) -> list:
        return [s for s in self.students if not s.is_valid()]

    def get_summary_stats(self) -> dict:
        total = len(self.students)
        if not total:
            return {}
        return {
            'total':       total,
            'passed':      sum(1 for s in self.students if s.status == 'PASS'),
            'failed':      sum(1 for s in self.students if s.status == 'FAIL'),
            'pending':     sum(1 for s in self.students if s.status == 'PENDING'),
            'invalid':     sum(1 for s in self.students if not s.is_valid()),
            'avg_grade':   round(sum(s.grade for s in self.students
                                     if s.grade is not None) /
                                  max(sum(1 for s in self.students if s.grade is not None), 1), 2),
            'total_fines': round(sum(s.late_fine for s in self.students), 2),
            'total_balance': round(sum(s.tuition_balance for s in self.students), 2),
        }

    def print_all(self):
        print(f'\n{" Student Records ":─^70}')
        for s in self.students:
            flag = ' ⚠' if not s.is_valid() else ''
            print('  ' + s.summary() + flag)
        print('─'*70)


print('✅ AcademicPortal controller defined')

✅ AcademicPortal controller defined


In [ ]:


class TerminalMenu:
    """Interactive terminal menu for the Academic Portal."""

    MENU = """
╔══════════════════════════════════════════════╗
║       UNIVERSITY ACADEMIC PORTAL             ║
╠══════════════════════════════════════════════╣
║  1. Load CSV & Run Full Pipeline             ║
║  2. View All Student Records                 ║
║  3. View Failed Students                     ║
║  4. View Validation Errors                   ║
║  5. View Summary Statistics                  ║
║  6. Generate Reports Only                    ║
║  7. Delete All Reports  [ADMIN]              ║
║  0. Exit                                     ║
╚══════════════════════════════════════════════╝"""

    def __init__(self):
        self.portal = AcademicPortal()

    def _require_loaded(self) -> bool:
        if not self.portal._loaded:
            print('⚠️  No data loaded. Please run option 1 first.')
            return False
        return True

    def option_1(self):
        filepath = input('📁 Enter CSV filepath [students.csv]: ').strip() or 'students.csv'
        try:
            self.portal.run_pipeline(filepath)
            print('\n✅ Pipeline complete!')
        except CSVLoadError as e:
            print(f'❌ Failed to load CSV: {e}')
        except Exception as e:
            print(f'❌ Unexpected error: {e}')

    def option_2(self):
        if self._require_loaded():
            self.portal.print_all()

    def option_3(self):
        if not self._require_loaded():
            return
        failed = self.portal.get_failed_students()
        print(f'\n── Failed Students ({len(failed)}) ─────────────────────')
        if not failed:
            print('   No failed students found.')
        for s in failed:
            print(f'  ❌ {s.summary()}')

    def option_4(self):
        if not self._require_loaded():
            return
        invalid = self.portal.get_invalid_records()
        print(f'\n── Validation Errors ({len(invalid)} records) ──────────────')
        if not invalid:
            print('   ✅ All records are valid.')
        for s in invalid:
            print(f'  [{s.student_id}] {s.name or "(no name)"}')
            for err in s.validation_errors:
                print(f'      ⚠ {err}')

    def option_5(self):
        if not self._require_loaded():
            return
        stats = self.portal.get_summary_stats()
        print('\n── Summary Statistics ──────────────────────────────')
        for k, v in stats.items():
            label = k.replace('_', ' ').title()
            val   = f'£{v:.2f}' if 'fine' in k or 'balance' in k else v
            print(f'   {label:<22}: {val}')

    def option_6(self):
        if not self._require_loaded():
            return
        async def _gen():
            return await self.portal.generate_reports()
        paths = asyncio.get_event_loop().run_until_complete(_gen())
        print('\n✅ Reports generated:')
        for p in paths:
            print(f'   📄 {p}')

    @validate_admin
    def option_7(self):
        report_files = list(Path(REPORT_DIR).glob('*.txt'))
        if not report_files:
            print('   No reports to delete.')
            return
        for f in report_files:
            f.unlink()
        print(f'🗑  Deleted {len(report_files)} report(s).')

    def run(self):
        print(f'\n🎓 Welcome to {PORTAL_NAME}')
        dispatch = {
            '1': self.option_1,
            '2': self.option_2,
            '3': self.option_3,
            '4': self.option_4,
            '5': self.option_5,
            '6': self.option_6,
            '7': self.option_7,
        }
        while True:
            print(self.MENU)
            choice = input('Select option: ').strip()
            if choice == '0':
                print('\n👋 Goodbye!\n')
                break
            elif choice in dispatch:
                print()
                dispatch[choice]()
                input('\n  [Press Enter to continue]')
            else:
                print('   Invalid option. Try again.')


print('✅ TerminalMenu defined')
print('\n━'*30)
print('✅ ALL COMPONENTS LOADED SUCCESSFULLY')
print('   Run the next cell to launch the portal!')
print('━'*30)

✅ TerminalMenu defined

━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
━
✅ ALL COMPONENTS LOADED SUCCESSFULLY
   Run the next cell to launch the portal!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:

menu = TerminalMenu()
menu.run()


🎓 Welcome to University Academic Portal

╔══════════════════════════════════════════════╗
║       UNIVERSITY ACADEMIC PORTAL             ║
╠══════════════════════════════════════════════╣
║  1. Load CSV & Run Full Pipeline             ║
║  2. View All Student Records                 ║
║  3. View Failed Students                     ║
║  4. View Validation Errors                   ║
║  5. View Summary Statistics                  ║
║  6. Generate Reports Only                    ║
║  7. Delete All Reports  [ADMIN]              ║
║  0. Exit                                     ║
╚══════════════════════════════════════════════╝


Select option:  1


📁 Enter CSV filepath [students.csv]:  



────────────────────────────────────────────────────────────
🎓 University Academic Portal — Processing Pipeline
────────────────────────────────────────────────────────────

[1/4] 📂 Loading CSV...
   📂 Loaded 10 records from students.csv
[2/4] 🔍 Validating records (async concurrent)...
   Found 1 invalid record(s) → errors logged to portal_errors.log
[3/4] 💰 Calculating late fines (async concurrent)...
   7 late submission(s), total fines: rupees92.50
[4/4] 📄 Generating reports (async concurrent)...
   ✅ Saved: reports\report_full_20260306_220247.txt
   ✅ Saved: reports\report_failed_only_20260306_220247.txt
   ✅ Saved: reports\report_fines_only_20260306_220247.txt
   ⏱  run_pipeline completed in 0.0206s

✅ Pipeline complete!



  [Press Enter to continue] 


...



────────────────────────── Student Records ───────────────────────────
  S001   | Alice Johnson        | Mathematics  | Grade:  78.0 | PASS    | Late: 2d | Fine: rupees5.00
  S002   | Bob Smith            | Physics      | Grade:  42.0 | FAIL    | Late: 5d | Fine: rupees12.50
  S003   | Carol White          | Chemistry    | Grade:  91.0 | PASS    | Late: 0d | Fine: None
  S004   | David Brown          | Biology      | Grade:  55.0 | PASS    | Late: 1d | Fine: rupees2.50 ⚠
  S005   | Eve Davis            | Mathematics  | Grade:  38.0 | FAIL    | Late: 10d | Fine: rupees25.00
  S006   | Frank Miller         | Physics      | Grade:  67.0 | PASS    | Late: 0d | Fine: None
  S007   | Grace Lee            | Chemistry    | Grade:   N/A | PENDING | Late: 2d | Fine: rupees5.00
  S008   | Henry Wilson         | Biology      | Grade:  80.0 | PASS    | Late: 2d | Fine: rupees5.00
  S009   | Ivy Moore            | Mathematics  | Grade:  29.0 | FAIL    | Late: 15d | Fine: rupees37.50
  S010   | Jac


  [Press Enter to continue] 0



╔══════════════════════════════════════════════╗
║       UNIVERSITY ACADEMIC PORTAL             ║
╠══════════════════════════════════════════════╣
║  1. Load CSV & Run Full Pipeline             ║
║  2. View All Student Records                 ║
║  3. View Failed Students                     ║
║  4. View Validation Errors                   ║
║  5. View Summary Statistics                  ║
║  6. Generate Reports Only                    ║
║  7. Delete All Reports  [ADMIN]              ║
║  0. Exit                                     ║
╚══════════════════════════════════════════════╝


Select option:  0



👋 Goodbye!



In [ ]:

print('🧪 Running automated demo...\n')

portal = AcademicPortal()
portal.run_pipeline('students.csv')

print('\n')
portal.print_all()

print('\n── Summary Stats ──────────────────────────')
for k, v in portal.get_summary_stats().items():
    label = k.replace('_', ' ').title()
    val   = f'£{v:.2f}' if 'fine' in k or 'balance' in k else v
    print(f'   {label:<22}: {val}')

print('\n── Failed Students ────────────────────────')
for s in portal.get_failed_students():
    print(f'   ❌ [{s.student_id}] {s.name} — Grade: {s.grade} — Fine: £{s.late_fine:.2f}')

print('\n── Invalid Records ────────────────────────')
for s in portal.get_invalid_records():
    print(f'   ⚠ [{s.student_id}] {s.name or "(no name)"}:')
    for err in s.validation_errors:
        print(f'      – {err}')

reports = list(Path(REPORT_DIR).glob('*.txt'))
print(f'\n── Reports Generated ({len(reports)}) ───────────────────')
for r in sorted(reports):
    size = r.stat().st_size
    print(f'   📄 {r.name}  ({size} bytes)')

🧪 Running automated demo...


────────────────────────────────────────────────────────────
🎓 University Academic Portal — Processing Pipeline
────────────────────────────────────────────────────────────

[1/4] 📂 Loading CSV...
   📂 Loaded 10 records from students.csv
[2/4] 🔍 Validating records (async concurrent)...
   Found 1 invalid record(s) → errors logged to portal_errors.log
[3/4] 💰 Calculating late fines (async concurrent)...
   7 late submission(s), total fines: rupees92.50
[4/4] 📄 Generating reports (async concurrent)...
   ✅ Saved: reports\report_full_20260306_220459.txt
   ✅ Saved: reports\report_failed_only_20260306_220459.txt
   ✅ Saved: reports\report_fines_only_20260306_220459.txt
   ⏱  run_pipeline completed in 0.0163s



────────────────────────── Student Records ───────────────────────────
  S001   | Alice Johnson        | Mathematics  | Grade:  78.0 | PASS    | Late: 2d | Fine: rupees5.00
  S002   | Bob Smith            | Physics      | Grade:  42.0 | FAIL    | Late: 